# CSMS AI Accuracy Improvement — Criterion-Level Reverse Engineering

**Goal:** Improve AI scoring accuracy by reverse-engineering human scoring patterns and expanding knowledge base.

## Approach:
1. **Re-evaluate all categories** using existing chunk databases
2. **Extract atomic facts** from chunks (document presence, field validity, implementation evidence)
3. **Map facts to criteria** (granular, not holistic)
4. **Compare against expected scores** (human-reviewed gold standard)
5. **Identify gaps** in knowledge base rules
6. **Generate candidate KB files** for review (separate from production)
7. **Test calibrated scoring** before merging to production

## Scoring Logic (New):
- Jurors can output any score (e.g., 2.5, 7.3)
- Final score uses **upward rounding**: 
  - `0` = no evidence
  - `(0, 3]` → `3`
  - `(3, 6]` → `6`  
  - `(6, 10]` → `10`

## Outputs:
- Criterion-level audit report per category
- Candidate KB files in `knowledge_base/marine/calibration/`
- Comparison: baseline vs calibrated accuracy
- HTML review dashboard

---

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import re
import sqlite3
import struct
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, Markdown, HTML

sys.path.append(os.path.abspath('..'))

PROJECT_ROOT = os.path.abspath('..')
MARINE_DIR = os.path.join(PROJECT_ROOT, 'test_submissions', 'marine')
KB_DIR = os.path.join(PROJECT_ROOT, 'knowledge_base', 'marine')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'notebook_v2', 'outputs')
DB_DIR = os.path.join(OUTPUT_DIR, 'databases')
CALIBRATION_DIR = os.path.join(KB_DIR, 'calibration')
REVIEW_DIR = os.path.join(OUTPUT_DIR, 'review')

os.makedirs(CALIBRATION_DIR, exist_ok=True)
os.makedirs(REVIEW_DIR, exist_ok=True)

BASELINE_RESULTS = os.path.join(OUTPUT_DIR, 'reports', 'batch_evaluation_results.json')

print(f'Project root: {PROJECT_ROOT}')
print(f'Marine dir:   {MARINE_DIR}')
print(f'KB dir:       {KB_DIR}')
print(f'Calibration:  {CALIBRATION_DIR}')
print(f'Review dir:   {REVIEW_DIR}')
print('\nSetup complete.')

In [ ]:
TARGET_CATEGORIES = None
USE_LARGE_DB = True
ENABLE_LLM_EXTRACTION = True
VERBOSE = True

if TARGET_CATEGORIES is None:
    print('Processing all 29 categories')
else:
    print(f'Processing {len(TARGET_CATEGORIES)} categories: {TARGET_CATEGORIES}')

print(f'Using database: {"large" if USE_LARGE_DB else "small"}')
print(f'LLM extraction: {ENABLE_LLM_EXTRACTION}')

## 2. Load Baseline Results

In [ ]:
with open(BASELINE_RESULTS, 'r', encoding='utf-8') as f:
    baseline = json.load(f)

baseline_by_id = {r['category_id']: r for r in baseline['results']}

print(f'Loaded baseline: {baseline["total_categories"]} categories')
print(f'Original accuracy: {baseline["summary"]["original"]["accuracy"]:.1f}%')
if 'enhanced_large' in baseline['summary']:
    print(f'Enhanced-large accuracy: {baseline["summary"]["enhanced_large"]["accuracy"]:.1f}%')

df_baseline = pd.DataFrame([
    {
        'category_id': r['category_id'],
        'expected': r['expected'],
        'original_pred': r['original']['predicted'],
        'original_correct': r['original_correct'],
        'enhanced_pred': r.get('enhanced_large', {}).get('predicted', 'N/A'),
        'enhanced_correct': r.get('enhanced_large_correct', 'N/A')
    }
    for r in baseline['results']
])
display(df_baseline)

## 3. Discover Categories & Load Knowledge Base

In [ ]:
def discover_categories(marine_dir: str) -> List[Dict[str, Any]]:
    categories = []
    pattern = re.compile(r'^([0-9]+(\.[0-9]+)?)')
    
    for item in sorted(os.listdir(marine_dir)):
        full_path = os.path.join(marine_dir, item)
        if os.path.isdir(full_path):
            m = pattern.match(item)
            if m:
                cat_id = m.group(1)
                expected_file = os.path.join(full_path, 'expected_result.json')
                expected_score = None
                if os.path.exists(expected_file):
                    with open(expected_file, 'r', encoding='utf-8') as f:
                        expected_score = json.load(f).get('expected_score')
                
                categories.append({
                    'id': cat_id,
                    'name': item,
                    'path': full_path,
                    'expected_score': expected_score
                })
    
    def sort_key(x):
        parts = x['id'].split('.')
        return [int(p) for p in parts]
    
    categories.sort(key=sort_key)
    return categories

def load_kb(category_id: str) -> Dict[str, Any]:
    clean_id = re.match(r'^[0-9]+(\.[0-9]+)?', str(category_id).strip())
    clean_id = clean_id.group(0) if clean_id else str(category_id).strip()
    result = {}
    for kind, sub, prefix in [('category', 'categories', ''), ('scoring_rules', 'scoring_rules', 'SR-'), ('validation_rules', 'validation_rules', 'VR-'), ('document_requirements', 'document_requirements', 'DR-')]:
        fpath = os.path.join(KB_DIR, sub, f'{prefix}{clean_id}.json')
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                result[kind] = json.load(f)
        else:
            result[kind] = {}
    return result

categories = discover_categories(MARINE_DIR)

if TARGET_CATEGORIES:
    categories = [c for c in categories if c['id'] in TARGET_CATEGORIES]

print(f'Discovered {len(categories)} categories')
for cat in categories[:5]:
    print(f'  {cat["id"]}: {cat["name"]} (expected: {cat["expected_score"]})')
if len(categories) > 5:
    print(f'  ... and {len(categories) - 5} more')

## 4. Database Functions

In [ ]:
def blob_to_embedding(blob: bytes) -> List[float]:
    n = len(blob) // 4
    return list(struct.unpack(f'{n}f', blob))

def load_category_chunks(category_id: str, use_large: bool = True) -> List[Dict[str, Any]]:
    db_suffix = 'large' if use_large else 'small'
    db_name = f'cat_{category_id.replace(".", "_")}_{db_suffix}.db'
    db_path = os.path.join(DB_DIR, db_name)
    
    if not os.path.exists(db_path):
        return []
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT chunk_id, category_id, filename, doc_type, tier, priority, 
               evidence_idx, chunk_idx, text, char_count, embedding
        FROM chunks
        ORDER BY priority, filename, evidence_idx, chunk_idx
    ''')
    
    rows = cursor.fetchall()
    conn.close()
    
    chunks = []
    for row in rows:
        chunks.append({
            'chunk_id': row[0],
            'category_id': row[1],
            'filename': row[2],
            'doc_type': row[3],
            'tier': row[4],
            'priority': row[5],
            'evidence_idx': row[6],
            'chunk_idx': row[7],
            'text': row[8],
            'char_count': row[9],
            'has_embedding': row[10] is not None
        })
    
    return chunks

print('Database functions loaded.')

## 5. Criterion-Level Fact Extraction

Extract atomic facts from chunks:
- **Document presence**: Which required documents exist?
- **Field validity**: Are mandatory fields present and valid?
- **Implementation evidence**: Is there proof of execution/compliance?
- **Temporal validity**: Are dates valid and not expired?
- **Completeness**: Are all required components present?

In [ ]:
def extract_document_facts(chunks: List[Dict[str, Any]], kb: Dict[str, Any]) -> Dict[str, Any]:
    facts = {
        'documents_present': [],
        'document_types': set(),
        'total_chunks': len(chunks),
        'total_chars': sum(c['char_count'] for c in chunks),
        'filenames': set(),
        'keywords_found': set(),
        'mandatory_fields_found': {},
        'dates_found': [],
        'signatures_mentioned': 0,
        'tables_detected': 0
    }
    
    vr = kb.get('validation_rules', {})
    keywords = vr.get('ai_checkpoint_keywords', [])
    mandatory_fields = vr.get('mandatory_fields', [])
    
    for chunk in chunks:
        text = chunk['text'].lower()
        facts['filenames'].add(chunk['filename'])
        facts['document_types'].add(chunk['doc_type'])
        
        for kw in keywords:
            if kw.lower() in text:
                facts['keywords_found'].add(kw)
        
        for field in mandatory_fields:
            field_lower = field.lower()
            if field_lower in text or any(part in text for part in field_lower.split()):
                facts['mandatory_fields_found'][field] = True
        
        import re
        date_patterns = [
            r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}',
            r'\d{4}[/-]\d{1,2}[/-]\d{1,2}',
            r'\d{1,2}\s+(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{2,4}'
        ]
        for pattern in date_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            facts['dates_found'].extend(matches)
        
        if any(kw in text for kw in ['signature', 'signed', 'tanda tangan', 'ditandatangani']):
            facts['signatures_mentioned'] += 1
        
        if 'table' in text or 'tabel' in text or '|' in chunk['text']:
            facts['tables_detected'] += 1
    
    facts['documents_present'] = list(facts['filenames'])
    facts['document_types'] = list(facts['document_types'])
    facts['keywords_found'] = list(facts['keywords_found'])
    
    return facts

def evaluate_criterion(criterion_type: str, facts: Dict[str, Any], kb: Dict[str, Any]) -> Dict[str, Any]:
    result = {
        'type': criterion_type,
        'status': 'unknown',
        'evidence': [],
        'confidence': 0.0
    }
    
    if criterion_type == 'document_presence':
        doc_count = len(facts['documents_present'])
        if doc_count == 0:
            result['status'] = 'fail'
            result['confidence'] = 1.0
        elif doc_count >= 3:
            result['status'] = 'pass'
            result['confidence'] = 0.9
        else:
            result['status'] = 'partial'
            result['confidence'] = 0.7
        result['evidence'] = facts['documents_present'][:5]
    
    elif criterion_type == 'keyword_coverage':
        vr = kb.get('validation_rules', {})
        required_keywords = vr.get('ai_checkpoint_keywords', [])
        found = facts['keywords_found']
        if required_keywords:
            coverage = len(found) / len(required_keywords)
            if coverage >= 0.8:
                result['status'] = 'pass'
                result['confidence'] = 0.85
            elif coverage >= 0.5:
                result['status'] = 'partial'
                result['confidence'] = 0.7
            else:
                result['status'] = 'fail'
                result['confidence'] = 0.8
        else:
            result['status'] = 'pass'
            result['confidence'] = 0.5
        result['evidence'] = found[:10]
    
    elif criterion_type == 'mandatory_fields':
        vr = kb.get('validation_rules', {})
        required = vr.get('mandatory_fields', [])
        found = facts['mandatory_fields_found']
        if required:
            coverage = len(found) / len(required)
            if coverage >= 0.8:
                result['status'] = 'pass'
                result['confidence'] = 0.85
            elif coverage >= 0.5:
                result['status'] = 'partial'
                result['confidence'] = 0.7
            else:
                result['status'] = 'fail'
                result['confidence'] = 0.8
        else:
            result['status'] = 'pass'
            result['confidence'] = 0.5
        result['evidence'] = list(found.keys())[:10]
    
    elif criterion_type == 'implementation_evidence':
        impl_keywords = ['implemented', 'executed', 'conducted', 'dilaksanakan', 'dilakukan', 'dijalankan']
        impl_count = sum(1 for chunk in facts.get('_chunks', []) if any(kw in chunk['text'].lower() for kw in impl_keywords))
        if impl_count >= 5:
            result['status'] = 'pass'
            result['confidence'] = 0.8
        elif impl_count >= 2:
            result['status'] = 'partial'
            result['confidence'] = 0.6
        else:
            result['status'] = 'fail'
            result['confidence'] = 0.7
        result['evidence'] = [f'{impl_count} implementation mentions']
    
    elif criterion_type == 'temporal_validity':
        date_count = len(facts['dates_found'])
        if date_count >= 3:
            result['status'] = 'pass'
            result['confidence'] = 0.75
        elif date_count >= 1:
            result['status'] = 'partial'
            result['confidence'] = 0.6
        else:
            result['status'] = 'fail'
            result['confidence'] = 0.7
        result['evidence'] = [f'{date_count} dates found']
    
    return result

def compute_criterion_score(criteria_results: List[Dict[str, Any]]) -> float:
    if not criteria_results:
        return 0.0
    
    status_weights = {'pass': 1.0, 'partial': 0.5, 'fail': 0.0, 'unknown': 0.0}
    
    weighted_sum = 0.0
    total_confidence = 0.0
    
    for crit in criteria_results:
        weight = status_weights.get(crit['status'], 0.0)
        confidence = crit.get('confidence', 0.5)
        weighted_sum += weight * confidence
        total_confidence += confidence
    
    if total_confidence == 0:
        return 0.0
    
    raw_score = (weighted_sum / total_confidence) * 10.0
    return raw_score

def snap_to_csms_scale(val: float) -> int:
    if val <= 0:
        return 0
    elif val <= 3:
        return 3
    elif val <= 6:
        return 6
    else:
        return 10

print('Criterion extraction functions loaded.')

## 6. Reverse-Engineering: All Categories

In [ ]:
category_audits = []

for cat in tqdm(categories, desc='Processing categories'):
    cat_id = cat['id']
    cat_name = cat['name']
    expected = cat['expected_score']
    
    if VERBOSE:
        print(f'\n{"="*80}')
        print(f'Category {cat_id}: {cat_name}')
        print(f'Expected score: {expected}')
    
    kb = load_kb(cat_id)
    chunks = load_category_chunks(cat_id, use_large=USE_LARGE_DB)
    
    if not chunks:
        if VERBOSE:
            print(f'  WARNING: No chunks found in database')
        category_audits.append({
            'category_id': cat_id,
            'category_name': cat_name,
            'expected': expected,
            'chunks': 0,
            'documents' :0,
            'keywords_found' : 0,
            'mandatory_fields_found':0,            
            'criteria': [],
            'raw_score':0,
            'calibrated_score': 0,
            'original_pred': 0,
            'enhanced_pred': 0,

            'gap_analysis': 'No chunks available'
        })
        continue
    
    facts = extract_document_facts(chunks, kb)
    facts['_chunks'] = chunks
    
    criteria_results = [
        evaluate_criterion('document_presence', facts, kb),
        evaluate_criterion('keyword_coverage', facts, kb),
        evaluate_criterion('mandatory_fields', facts, kb),
        evaluate_criterion('implementation_evidence', facts, kb),
        evaluate_criterion('temporal_validity', facts, kb)
    ]
    
    raw_score = compute_criterion_score(criteria_results)
    calibrated_score = snap_to_csms_scale(raw_score)
    
    baseline_result = baseline_by_id.get(cat_id, {})
    original_pred = baseline_result.get('original', {}).get('predicted', 'N/A')
    enhanced_pred = baseline_result.get('enhanced_large', {}).get('predicted', 'N/A')
    
    gap_analysis = []
    if calibrated_score < expected:
        for crit in criteria_results:
            if crit['status'] in ['fail', 'partial']:
                gap_analysis.append(f'{crit["type"]}: {crit["status"]}')
    
    audit = {
        'category_id': cat_id,
        'category_name': cat_name,
        'expected': expected,
        'chunks': len(chunks),
        'documents': len(facts['documents_present']),
        'keywords_found': len(facts['keywords_found']),
        'mandatory_fields_found': len(facts['mandatory_fields_found']),
        'criteria': criteria_results,
        'raw_score': round(raw_score, 2),
        'calibrated_score': calibrated_score,
        'original_pred': original_pred,
        'enhanced_pred': enhanced_pred,
        'gap_analysis': gap_analysis
    }
    category_audits.append(audit)
    
    if VERBOSE:
        print(f'  Chunks: {len(chunks)}')
        print(f'  Documents: {len(facts["documents_present"])}')
        print(f'  Keywords: {len(facts["keywords_found"])}')
        print(f'  Raw score: {raw_score:.2f} → Calibrated: {calibrated_score}')
        print(f'  Original: {original_pred}, Enhanced: {enhanced_pred}')
        if gap_analysis:
            print(f'  Gaps: {", ".join(gap_analysis)}')

print(f'\n{"="*80}')
print(f'Processed {len(category_audits)} categories')

## 7. Results Summary

In [ ]:
category_audits

In [ ]:
df_audits = pd.DataFrame([
    {
        'category_id': a['category_id'],
        'category_name': a['category_name'][:50],
        'expected': a['expected'],
        'chunks': a['chunks'],
        'documents': a['documents'] ,
        'keywords': a['keywords_found'],
        'mandatory': a['mandatory_fields_found'],
        'raw_score': a['raw_score'],
        'calibrated': a['calibrated_score'],
        'original': a['original_pred'],
        'enhanced': a['enhanced_pred'],
        'calibrated_correct': int(a['calibrated_score'] == a['expected']),
        'gaps': len(a['gap_analysis']) if isinstance(a['gap_analysis'], list) else 0
    }
    for a in category_audits
])

display(df_audits)

calibrated_correct = df_audits['calibrated_correct'].sum()
calibrated_accuracy = calibrated_correct / len(df_audits) * 100

print(f'\n{"="*80}')
print(f'CALIBRATED SCORING ACCURACY: {calibrated_correct}/{len(df_audits)} ({calibrated_accuracy:.1f}%)')
print(f'BASELINE ORIGINAL: {baseline["summary"]["original"]["accuracy"]:.1f}%')
if 'enhanced_large' in baseline['summary']:
    print(f'BASELINE ENHANCED: {baseline["summary"]["enhanced_large"]["accuracy"]:.1f}%')
print(f'IMPROVEMENT: +{calibrated_accuracy - baseline["summary"]["original"]["accuracy"]:.1f}%')
print(f'{"="*80}')

## 8. Gap Analysis & KB Recommendations

In [ ]:
gap_summary = {}
for audit in category_audits:
    if isinstance(audit['gap_analysis'], list):
        for gap in audit['gap_analysis']:
            criterion_type = gap.split(':')[0]
            gap_summary[criterion_type] = gap_summary.get(criterion_type, 0) + 1

print('=== GAP ANALYSIS ===\n')
print('Criterion failures across all categories:')
for criterion, count in sorted(gap_summary.items(), key=lambda x: x[1], reverse=True):
    print(f'  {criterion}: {count} failures')

print('\n=== RECOMMENDATIONS ===\n')
print('1. DOCUMENT PRESENCE:')
print('   - Expand document_requirements/*.json with specific file patterns')
print('   - Add marine-specific document types (vessel logs, port records)')
print()
print('2. KEYWORD COVERAGE:')
print('   - Add marine terminology to ai_checkpoint_keywords')
print('   - Include Indonesian/English variations')
print('   - Add regulatory references (SOLAS, MARPOL, ISPS, STCW)')
print()
print('3. MANDATORY FIELDS:')
print('   - Convert free-text descriptions to structured field specs')
print('   - Add field validation rules (format, range, required)')
print()
print('4. IMPLEMENTATION EVIDENCE:')
print('   - Add criteria for execution proof (photos, logs, certificates)')
print('   - Define minimum evidence count per category')
print()
print('5. TEMPORAL VALIDITY:')
print('   - Add date extraction and validation rules')
print('   - Check certificate expiry dates')
print('   - Verify document currency (last 3 years)')

if gap_summary:
    print('\n=== PRIORITY CATEGORIES FOR KB EXPANSION ===\n')
    priority_cats = [a for a in category_audits if a['calibrated_score'] < a['expected']]
    priority_cats.sort(key=lambda x: x['expected'] - x['calibrated_score'], reverse=True)
    for cat in priority_cats[:10]:
        gap_count = len(cat['gap_analysis']) if isinstance(cat['gap_analysis'], list) else 0
        print(f'  {cat["category_id"]} (expected {cat["expected"]}, got {cat["calibrated_score"]}): {gap_count} gaps')

## 9. Generate Candidate KB Files

In [ ]:
def generate_candidate_kb(audit: Dict[str, Any], kb: Dict[str, Any]) -> Dict[str, Any]:
    cat_id = audit['category_id']
    expected = audit['expected']
    
    candidate = {
        'category_id': cat_id,
        'expected_score': expected,
        'calibrated_score': audit['calibrated_score'],
        'criteria': audit['criteria'],
        'kb_expansion': {
            'additional_keywords': [],
            'additional_mandatory_fields': [],
            'additional_document_requirements': [],
            'implementation_checks': []
        },
        'score_gates': {
            '0': {'min_documents': 0, 'min_keywords': 0, 'min_mandatory': 0},
            '3': {'min_documents': 1, 'min_keywords': 2, 'min_mandatory': 1},
            '6': {'min_documents': 3, 'min_keywords': 4, 'min_mandatory': 3},
            '10': {'min_documents': 5, 'min_keywords': 6, 'min_mandatory': 5, 'implementation_proof': True}
        }
    }
    
    for crit in audit['criteria']:
        if crit['status'] == 'fail' or crit['status'] == 'partial':
            if crit['type'] == 'keyword_coverage':
                candidate['kb_expansion']['additional_keywords'].extend([
                    f'ADD: marine-specific term for {cat_id}',
                    f'ADD: regulatory reference (SOLAS/MARPOL/ISPS/STCW)',
                    f'ADD: Indonesian/English variation'
                ])
            elif crit['type'] == 'mandatory_fields':
                candidate['kb_expansion']['additional_mandatory_fields'].extend([
                    f'ADD: structured field spec for {cat_id}',
                    f'ADD: validation rule (format, range)'
                ])
            elif crit['type'] == 'document_presence':
                candidate['kb_expansion']['additional_document_requirements'].extend([
                    f'ADD: marine document type for {cat_id}',
                    f'ADD: file pattern matching'
                ])
            elif crit['type'] == 'implementation_evidence':
                candidate['kb_expansion']['implementation_checks'].extend([
                    f'ADD: execution proof requirement for {cat_id}',
                    f'ADD: minimum evidence count'
                ])
    
    return candidate

candidates = []
for audit in category_audits:
    if audit['calibrated_score'] < audit['expected']:
        kb = load_kb(audit['category_id'])
        candidate = generate_candidate_kb(audit, kb)
        candidates.append(candidate)
        
        candidate_path = os.path.join(CALIBRATION_DIR, f'candidate_{audit["category_id"].replace(".", "_")}.json')
        with open(candidate_path, 'w', encoding='utf-8') as f:
            json.dump(candidate, f, indent=2, ensure_ascii=False)

print(f'Generated {len(candidates)} candidate KB files in: {CALIBRATION_DIR}')
print(f'\nCandidate files created for categories where calibrated_score < expected_score')
print(f'Review these files and merge valid expansions to production KB')

## 10. Save Audit Results

In [ ]:
def default_serialized(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)
    

audit_report = {
    'timestamp': datetime.now().isoformat(),
    'total_categories': len(category_audits),
    'configuration': {
        'use_large_db': USE_LARGE_DB,
        'enable_llm_extraction': ENABLE_LLM_EXTRACTION,
        'scoring_logic': 'upward_rounding'
    },
    'summary': {
        'calibrated_accuracy': float(calibrated_accuracy),
        'calibrated_correct': int(calibrated_correct),
        'baseline_original_accuracy': float(baseline['summary']['original']['accuracy']),
        'improvement': float(calibrated_accuracy - baseline['summary']['original']['accuracy'])
    },
    'gap_analysis': gap_summary,
    'audits': category_audits,
    'candidates_generated': len(candidates)
}

audit_path = os.path.join(REVIEW_DIR, 'criterion_level_audit.json')
with open(audit_path, 'w', encoding='utf-8') as f:
    json.dump(audit_report, f, indent=2, ensure_ascii=False, default = default_serialized)

df_audits.to_csv(os.path.join(REVIEW_DIR, 'criterion_level_audit.csv'), index=False)

print(f'Audit saved:')
print(f'  JSON: {audit_path}')
print(f'  CSV:  {os.path.join(REVIEW_DIR, "criterion_level_audit.csv")}')
print(f'\nFile sizes:')
print(f'  JSON: {os.path.getsize(audit_path) / 1024:.1f} KB')
print(f'  CSV:  {os.path.getsize(os.path.join(REVIEW_DIR, "criterion_level_audit.csv")) / 1024:.1f} KB')

## 11. Next Steps

### Immediate Actions:
1. **Review candidate KB files** in `knowledge_base/marine/calibration/`
2. **Validate recommendations** with domain experts
3. **Merge valid expansions** to production KB (categories/, validation_rules/, scoring_rules/)
4. **Re-run batch evaluation** to measure improvement

### KB Expansion Priorities:
1. Add marine-specific keywords to `ai_checkpoint_keywords`
2. Expand `mandatory_fields` with structured field specs
3. Add implementation evidence requirements
4. Include regulatory references (SOLAS, MARPOL, ISPS, STCW)
5. Define score gates with minimum evidence thresholds

### Validation:
- Test calibrated scoring on holdout categories
- Compare against human peer review
- Iterate KB expansion based on failures

### Files Generated:
- `outputs/review/criterion_level_audit.json` — Complete audit report
- `outputs/review/criterion_level_audit.csv` — Tabular summary
- `knowledge_base/marine/calibration/candidate_*.json` — Per-category KB expansion candidates

### Production Merge Process:
1. Review each `candidate_*.json` file
2. Validate KB expansion recommendations
3. Copy valid expansions to production KB
4. Re-run batch evaluation to confirm improvement
5. Commit changes with audit trail